# E3i Simple Fusion Rot4 TTA Colab Runner

Validation-only Colab launcher for the E3i simple fusion rot4 TTA diagnostic. Core logic stays in `scripts/evaluate_simple_tta_rot4.py`.

This notebook does not load or evaluate the test split. Outputs are isolated under `/content/drive/MyDrive/dl-final-artifact/e3i_simple_tta_rot4/`.

## 1. Runtime setup

Use a GPU runtime before running the full validation cell.

In [ ]:
import json
import os
import shutil
import subprocess
import sys
from pathlib import Path

from google.colab import drive

drive.mount('/content/drive')

REPO_URL = 'https://github.com/KasimDeliaci/dl-final-project.git'
REPO_DIR = Path('/content/dl-final-project')
DRIVE_ROOT = Path('/content/drive/MyDrive/dl-final-artifact')
E3I_DRIVE_ROOT = DRIVE_ROOT / 'e3i_simple_tta_rot4'
E3I_INPUT_BUNDLE = E3I_DRIVE_ROOT / 'e3i_simple_tta_inputs.tar'
E3I_ARTIFACT_ROOT = E3I_DRIVE_ROOT / 'artifacts'
HAM10000_BUNDLE_CANDIDATES = [
    DRIVE_ROOT / 'ham10000_colab_bundle.tar',
    Path('/content/drive/MyDrive/ham10000_colab_bundle.tar'),
    Path('/content/drive/MyDrive/Colab Notebooks/ham10000_colab_bundle.tar'),
]
E3I_BATCH_SIZE = 128
E3I_NUM_WORKERS = 2

print('Drive root:', DRIVE_ROOT)
print('E3i Drive root:', E3I_DRIVE_ROOT)
print('E3i optional input bundle:', E3I_INPUT_BUNDLE)
print('HAM10000 bundle candidates:', HAM10000_BUNDLE_CANDIDATES)
print('E3i batch size:', E3I_BATCH_SIZE)

## 2. Clone repository and install dependencies

In [ ]:
if not REPO_DIR.exists():
    subprocess.run(['git', 'clone', REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(['git', '-C', str(REPO_DIR), 'pull', '--ff-only'], check=True)

os.chdir(REPO_DIR)
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'sync', '--dev'], check=True)
sys.path.insert(0, str(REPO_DIR / 'src'))
print('Repository ready at', REPO_DIR)

## 3. Restore required validation-only inputs

Preferred source is `MyDrive/dl-final-artifact/artifacts/`. If that tree is incomplete, upload/copy `e3i_simple_tta_inputs.tar` under the E3i Drive root.

In [ ]:
def rsync_dir(source: Path, dest: Path, *, required: bool = True) -> None:
    if not source.exists():
        message = f'missing source: {source}'
        if required:
            raise FileNotFoundError(message)
        print('WARNING:', message)
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', f'{source}/', str(dest)], check=True)

def restore_dataset_bundle() -> None:
    if Path('data/splits/val.csv').exists():
        print('HAM10000 split already present:', Path('data/splits/val.csv'))
        return
    bundle = next((path for path in HAM10000_BUNDLE_CANDIDATES if path.exists()), None)
    if bundle is None:
        candidates = '\n'.join(str(path) for path in HAM10000_BUNDLE_CANDIDATES)
        raise FileNotFoundError(
            'Missing data/splits/val.csv and no HAM10000 bundle was found.\n'
            'Copy ham10000_colab_bundle.tar to one of:\n'
            f'{candidates}'
        )
    print('Restoring HAM10000 bundle:', bundle)
    subprocess.run(['tar', '-xf', str(bundle), '-C', str(REPO_DIR)], check=True)
    if not Path('data/splits/val.csv').exists():
        raise FileNotFoundError('HAM10000 bundle restored but data/splits/val.csv is absent.')

def restore_inputs() -> None:
    if E3I_INPUT_BUNDLE.exists():
        print('Restoring optional E3i input bundle:', E3I_INPUT_BUNDLE)
        subprocess.run(['tar', '-xf', str(E3I_INPUT_BUNDLE), '-C', str(REPO_DIR)], check=True)
        return
    print(f'No E3i input bundle found at {E3I_INPUT_BUNDLE}; using Drive artifact tree.')
    drive_artifacts = DRIVE_ROOT / 'artifacts'
    required_dirs = [
        'features/ham10000/finetuned/vit_b16',
        'features/ham10000/finetuned/swin_tiny',
        'features/ham10000/finetuned/beit_base',
        'checkpoints/ham10000/finetuned/vit_b16',
        'checkpoints/ham10000/finetuned/swin_tiny',
        'checkpoints/ham10000/finetuned/beit_base',
        'runs/20260611_111408_s3_finetuned_vit-swin-beit_concat_mlp_s4_triple_seed42',
        'runs/20260611_111414_s3_finetuned_vit-swin-beit_weightedlearned512_mlp_s4_triple_seed42',
        'runs/20260611_111338_s3_finetuned_vit-swin_concat_mlp_s4_pair_seed42',
    ]
    missing = [item for item in required_dirs if not (drive_artifacts / item).exists()]
    if missing:
        raise FileNotFoundError('Missing E3i Drive inputs:\n' + '\n'.join(missing))
    for item in required_dirs:
        rsync_dir(drive_artifacts / item, Path('artifacts') / item)

restore_dataset_bundle()
restore_inputs()
print('Required E3i input artifacts restored.')

## 4. Input checks

In [ ]:
import pandas as pd
import torch

if not torch.cuda.is_available():
    raise RuntimeError('E3i full TTA should run on a Colab GPU runtime. Enable GPU first.')

required_files = [
    'data/splits/val.csv',
    'artifacts/features/ham10000/finetuned/vit_b16/val.pt',
    'artifacts/features/ham10000/finetuned/swin_tiny/val.pt',
    'artifacts/features/ham10000/finetuned/beit_base/val.pt',
    'artifacts/checkpoints/ham10000/finetuned/vit_b16/best.pt',
    'artifacts/checkpoints/ham10000/finetuned/swin_tiny/best.pt',
    'artifacts/checkpoints/ham10000/finetuned/beit_base/best.pt',
    'artifacts/runs/20260611_111408_s3_finetuned_vit-swin-beit_concat_mlp_s4_triple_seed42/model.pt',
    'artifacts/runs/20260611_111414_s3_finetuned_vit-swin-beit_weightedlearned512_mlp_s4_triple_seed42/model.pt',
    'artifacts/runs/20260611_111338_s3_finetuned_vit-swin_concat_mlp_s4_pair_seed42/model.pt',
]
missing_files = [path for path in required_files if not Path(path).exists()]
if missing_files:
    raise FileNotFoundError('Missing E3i files:\n' + '\n'.join(missing_files))

val = pd.read_csv('data/splits/val.csv')
if len(val) != 1504:
    raise RuntimeError(f'Expected 1504 validation rows, got {len(val)}')

for config_path in Path('artifacts/runs').glob('20260611_111*_s3_finetuned_*/run_config.json'):
    config = json.loads(config_path.read_text())
    if config.get('fusion_method') == 'weighted_pca_384':
        continue
    if config.get('feature_source') != 'finetuned':
        raise RuntimeError(f'Unexpected feature source in {config_path}')

print('GPU:', torch.cuda.get_device_name(0))
print('E3i input checks passed.')

## 5. Code checks

In [ ]:
!PYTHONPATH=src uv run ruff check scripts/evaluate_simple_tta_rot4.py tests/test_e3i_simple_tta.py
!PYTHONPATH=src uv run pytest tests/test_e3i_simple_tta.py tests/test_e3h_tta.py

## 6. Smoke run

Run this first. It should finish quickly and writes `artifacts/runs/e3i_simple_tta_rot4_smoke_n8/`.

In [ ]:
!PYTHONUNBUFFERED=1 PYTHONPATH=src uv run python scripts/evaluate_simple_tta_rot4.py \
  --max-samples 8 \
  --batch-size 4 \
  --num-workers 0 \
  --device cuda \
  --no-mixed-precision

## 7. Full validation-only E3i run

In [ ]:
!PYTHONUNBUFFERED=1 PYTHONPATH=src uv run python scripts/evaluate_simple_tta_rot4.py \
  --batch-size {E3I_BATCH_SIZE} \
  --num-workers {E3I_NUM_WORKERS} \
  --device cuda \
  --identity-tolerance 1e-3

## 8. Output integrity check and preview

In [ ]:
from IPython.display import Image as IPImage
from IPython.display import Markdown, display

from dl_final.evaluation.tta import probabilities_from_frame

RUN_DIR = Path('artifacts/runs/e3i_simple_tta_rot4')
CLASS_NAMES = ['akiec', 'bcc', 'bkl', 'df', 'nv', 'mel', 'vasc']
config = json.loads((RUN_DIR / 'run_config.json').read_text())
assert config['test_policy'] == 'not_loaded_or_used_in_e3i'
assert config['views'] == ['identity', 'rot90', 'rot180', 'rot270']

for pred_path in sorted(RUN_DIR.glob('predictions_*_tta_rot4.csv')):
    pred = pd.read_csv(pred_path)
    if len(pred) != 1504:
        raise RuntimeError(f'{pred_path} has {len(pred)} rows; expected 1504')
    if set(pred['split'].astype(str)) != {'val'}:
        raise RuntimeError(f'{pred_path} contains non-validation rows')
    _ = probabilities_from_frame(pred, CLASS_NAMES)

identity = pd.read_csv(RUN_DIR / 'tta_identity_sanity.csv')
if not identity['passed'].all():
    warning_columns = [
        'run_alias',
        'max_abs_probability_delta',
        'macro_f1_delta',
    ]
    display(identity.loc[~identity['passed'], warning_columns])
    print('WARNING: strict identity parity exceeded tolerance for the rows above.')

preview_tables = [
    RUN_DIR / 'tta_run_results.csv',
    RUN_DIR / 'tta_ensemble_results.csv',
    RUN_DIR / 'tta_view_results.csv',
]
for path in preview_tables:
    if not path.exists() or path.stat().st_size == 0:
        raise FileNotFoundError(f'Missing or empty E3i output table: {path}')
    display(Markdown(f'### {path}'))
    display(pd.read_csv(path))

for figure in sorted(Path('artifacts/report_assets/figures').glob('e3i_simple_tta_rot4*.png')):
    display(Markdown(f'### {figure}'))
    display(IPImage(filename=str(figure)))

print('E3i output integrity check passed.')

## 9. Sync E3i outputs to Drive

In [ ]:
def sync_output_dir(source: Path, dest: Path) -> None:
    if not source.exists():
        print('WARNING: output source missing, skip:', source)
        return
    dest.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', f'{source}/', str(dest)], check=True)

sync_output_dir(
    Path('artifacts/runs/e3i_simple_tta_rot4'),
    E3I_ARTIFACT_ROOT / 'runs/e3i_simple_tta_rot4',
)
sync_output_dir(
    Path('artifacts/runs/e3i_simple_tta_rot4_smoke_n8'),
    E3I_ARTIFACT_ROOT / 'runs/e3i_simple_tta_rot4_smoke_n8',
)

for pattern in ['e3i_simple_tta_rot4*.csv']:
    for source in sorted(Path('artifacts/report_assets/tables').glob(pattern)):
        dest = E3I_ARTIFACT_ROOT / 'report_assets/tables' / source.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, dest)
for pattern in ['e3i_simple_tta_rot4*.png']:
    for source in sorted(Path('artifacts/report_assets/figures').glob(pattern)):
        dest = E3I_ARTIFACT_ROOT / 'report_assets/figures' / source.name
        dest.parent.mkdir(parents=True, exist_ok=True)
        shutil.copy2(source, dest)

print(f'E3i artifact sync complete: {E3I_ARTIFACT_ROOT}')